In [1]:
# =============================================================
# Korean 3-Class Selection + HF Tokenization Pipeline
# Google Colab Version
# =============================================================

# -------------------------------------------------------------
# 0. Setup — Colab 패키지 설치
# -------------------------------------------------------------
# Colab 최초 실행 시 아래 셀을 먼저 실행
!pip install -q transformers scikit-learn openpyxl


In [2]:

import os
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
import math

SEED = 42
TOP_K = 10
np.random.seed(SEED)
os.makedirs("figs", exist_ok=True)



In [5]:

# =============================================================
# Part A · Data Selection (3-Class Balanced)
# =============================================================

# 1. Load Dataset — 한국어만 사용
url = "https://raw.githubusercontent.com/hongsukyi/Lectures/main/data/aihub_conver.xlsx"

df_orig = pd.read_excel(url, engine="openpyxl")

df_orig.head(10)


,대분류,소분류,상황,Set Nr.,발화자,원문,번역문
0,비즈니스,회의,의견 교환하기,1,A-1,이번 신제품 출시에 대한 시장의 반응은 어떤가요?,How is the market's reaction to the newly rele...
1,비즈니스,회의,의견 교환하기,1,B-1,판매량이 지난번 제품보다 빠르게 늘고 있습니다.,The sales increase is faster than the previous...
2,비즈니스,회의,의견 교환하기,1,A-2,그렇다면 공장에 연락해서 주문량을 더 늘려야겠네요.,"Then, we'll have to call the manufacturer and ..."
3,비즈니스,회의,의견 교환하기,1,B-2,"네, 제가 연락해서 주문량을 2배로 늘리겠습니다.","Sure, I'll make a call and double the volume o..."
4,비즈니스,회의,의견 교환하기,2,A-1,지난 회의 마지막에 논의했던 안건을 다시 볼까요?,Shall we take a look at the issues we discusse...
5,비즈니스,회의,의견 교환하기,2,B-1,그보다는 이번 주 새로운 주제가 더 급한 것 같습니다.,I believe that this week's new issues are more...
6,비즈니스,회의,의견 교환하기,2,A-2,그럼 새로운 안건으로 회의를 시작하도록 하죠.,"Then, let's begin our meeting with the new iss..."
7,비즈니스,회의,의견 교환하기,2,B-2,"네, 자료는 여러분의 앞에 미리 준비되어 있습니다.","Sure, the related materials are ready in front..."
8,비즈니스,회의,의견 교환하기,3,A-1,이번 주 금요일까지 2천개를 더 주문하라는 건가요?,"Do you mean we need to order additional 2,000 ..."
9,비즈니스,회의,의견 교환하기,3,B-1,"네, 시간이 조금 촉박하기는 하지만 가능해 보이는데요.","Yes, time is running short, but we can manage it."


In [6]:
df_orig.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   대분류      100000 non-null  object
 1   소분류      100000 non-null  object
 2   상황       100000 non-null  object
 3   Set Nr.  100000 non-null  int64 
 4   발화자      100000 non-null  object
 5   원문       100000 non-null  object
 6   번역문      100000 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [7]:
df_orig.describe()

,Set Nr.
count,100000.000000
mean,12500.500000
std,7216.914444
min,1.000000
25%,6250.750000
50%,12500.500000
75%,18750.250000
max,25000.000000


In [9]:

df_raw = df_orig[["원문", "상황"]].dropna()

print(f"Total samples: {len(df_raw):,}")

df_raw.head(10)

Total samples: 100,000


,원문,상황
0,이번 신제품 출시에 대한 시장의 반응은 어떤가요?,의견 교환하기
1,판매량이 지난번 제품보다 빠르게 늘고 있습니다.,의견 교환하기
2,그렇다면 공장에 연락해서 주문량을 더 늘려야겠네요.,의견 교환하기
3,"네, 제가 연락해서 주문량을 2배로 늘리겠습니다.",의견 교환하기
4,지난 회의 마지막에 논의했던 안건을 다시 볼까요?,의견 교환하기
5,그보다는 이번 주 새로운 주제가 더 급한 것 같습니다.,의견 교환하기
6,그럼 새로운 안건으로 회의를 시작하도록 하죠.,의견 교환하기
7,"네, 자료는 여러분의 앞에 미리 준비되어 있습니다.",의견 교환하기
8,이번 주 금요일까지 2천개를 더 주문하라는 건가요?,의견 교환하기
9,"네, 시간이 조금 촉박하기는 하지만 가능해 보이는데요.",의견 교환하기


In [10]:
# 2. Top-K Domain Check
label_counts = df_raw["상황"].value_counts()
top_k = label_counts.head(TOP_K)

print(f"\nTop {TOP_K} Classes")
for i, (label, cnt) in enumerate(top_k.items(), 1):
    print(f"{i:2d}. {label} ({cnt:,})")




Top 10 Classes
 1. 직장에서의 일상 대화 (상사, 동료, 부하 간의 대화) (1,320)
 2. 찬성 및 반대 (1,076)
 3. 취직 면접 상황 (1,032)
 4. 회의 관련 (1,020)
 5. 의견 교환하기 (1,016)
 6. 학교생활 (시험, 졸업, 입학 등) (976)
 7. 원하는 스타일에 대해 점원 or 친구와 대화 시술 전/시술 시/시술 후 대화 (948)
 8. 제안 및 협상하기 (924)
 9. CS/고객 상담 (고객 요청 대응: 취소, 반품, 결함 등) (832)
10. 마케팅/홍보 (804)


In [11]:

# -------------------------------------------------------------
# 3. Class Name Simplification
# -------------------------------------------------------------
label_map = {
    "직장에서의 일상 대화 (상사, 동료, 부하 간의 대화)": "직장",
    "찬성 및 반대": "찬반",
    "취직 면접 상황": "면접",
    "회의 관련": "회의",
    "의견 교환하기": "의견교환",
    "학교생활 (시험, 졸업, 입학 등)": "학교",
    "원하는 스타일에 대해 점원 or 친구와 대화 시술 전/시술 시/시술 후 대화": "시술상담",
    "제안 및 협상하기": "협상",
    "CS/고객 상담 (고객 요청 대응: 취소, 반품, 결함 등)": "고객",
    "마케팅/홍보": "마케팅",
}

df_raw["label_simple"] = df_raw["상황"].map(label_map)

n_unmapped = df_raw["label_simple"].isna().sum()
print(f"\nUnmapped labels: {n_unmapped:,}")



Unmapped labels: 90,052


In [12]:

SELECT_LABELS = ["면접", "학교", "협상"]

df_sel = df_raw[df_raw["label_simple"].isin(SELECT_LABELS)].copy()

assert len(df_sel) > 0, \
    f"[ERROR] SELECT_LABELS {SELECT_LABELS} 에 해당하는 데이터가 없습니다."

print("\nSelected Classes:")
for lb in SELECT_LABELS:
    print(f"  - {lb} ({(df_sel['label_simple'] == lb).sum():,})")




Selected Classes:
  - 면접 (1,032)
  - 학교 (976)
  - 협상 (924)


In [13]:

# -------------------------------------------------------------
# 4. Balanced Sampling
# -------------------------------------------------------------
SAMP = min(df_sel["label_simple"].value_counts().min(), 800)

df_clf = (
    df_sel.groupby("label_simple")
    .sample(n=SAMP, random_state=SEED)
    .reset_index(drop=True)
)

print("\nBalanced dataset size:", len(df_clf))



Balanced dataset size: 2400


In [14]:


# =============================================================
# Part B · HuggingFace Tokenization
# =============================================================

# 5. Prepare Dataset
df = df_clf[["원문", "label_simple"]].copy()
df.rename(columns={"원문": "text", "label_simple": "label"}, inplace=True)

print("Final dataset size:", len(df))
print("-" * 60)


Final dataset size: 2400
------------------------------------------------------------


In [15]:


# 6. Load Tokenizer
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print("-" * 60)



config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizer: klue/bert-base
Vocab size: 32,000
------------------------------------------------------------


In [16]:

# -------------------------------------------------------------
# 7. Automatic MAX_LEN (p95 기반)
# -------------------------------------------------------------
lengths = [len(tokenizer.tokenize(t)) for t in df["text"]]
p95, p99 = np.percentile(lengths, [95, 99])

special_tokens_count = tokenizer.num_special_tokens_to_add(pair=False)
margin = 2

raw_max_len = int(p95 + special_tokens_count + margin)
MAX_LEN = int(math.ceil(raw_max_len / 8) * 8)

print(f"p95: {p95:.0f}, p99: {p99:.0f}")
print(f"special tokens: {special_tokens_count}")
print(f"자동 설정 MAX_LEN: {MAX_LEN}")
print("-" * 60)


p95: 25, p99: 30
special tokens: 2
자동 설정 MAX_LEN: 32
------------------------------------------------------------


In [17]:


# -------------------------------------------------------------
# 8. Batch Tokenization
# -------------------------------------------------------------
encoded = tokenizer(
    df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors=None
)

input_ids      = np.array(encoded["input_ids"],      dtype=np.int32)
attention_mask = np.array(encoded["attention_mask"], dtype=np.int32)

pad_id = tokenizer.pad_token_id

print(f"Encoded shape : {input_ids.shape}")
print("-" * 60)



Encoded shape : (2400, 32)
------------------------------------------------------------


In [18]:

# =============================================================
# Part C · Label Encoding + Single File Save
# =============================================================

# 9. Label Encoding
le = LabelEncoder()
labels = le.fit_transform(df["label"])
class_names = le.classes_.tolist()

print("Class names:", class_names)



Class names: ['면접', '학교', '협상']


In [19]:

# 10. Save — 단일 파일 완결형
n_total = len(df_clf)
fname = f"aihub_enko_conv_kluebert_3class_n{n_total}_v1.npz"

np.savez(
    fname,
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    vocab_size=np.int32(tokenizer.vocab_size),
    max_len=np.int32(MAX_LEN),
    pad_id=np.int32(pad_id),
    num_classes=np.int32(len(class_names)),
    class_names=np.array(class_names)
)

print(f"\nSaved: {fname}")




Saved: aihub_enko_conv_kluebert_3class_n2400_v1.npz
